In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
import plotly.graph_objects as go

from datasets import load_dataset

# Load a well-known dataset, for example, 'imdb' for sentiment analysis
# This will download and cache the dataset if it's not already on your system.

model = SentenceTransformer("all-MiniLM-L6-v2")

In [2]:
import plotly.express as px
import random

def get_random_discrete_colors(n_colors, palette_name=None):
    """
    Returns a list of n_colors from a random Plotly qualitative color palette,
    or a specified palette.
    """
    qualitative_palettes = [
        "Plotly", "D3", "G10", "T10", "Alphabet", "Light24", "Dark24",
        "Set1", "Set2", "Set3", "Pastel1", "Pastel2", "Vivid"
    ]

    if palette_name:
        if palette_name not in qualitative_palettes:
            raise ValueError(f"Palette '{palette_name}' not found. Available: {qualitative_palettes}")
        chosen_palette = getattr(px.colors.qualitative, palette_name)
    else:
        # Choose a random qualitative palette
        chosen_palette_name = random.choice(qualitative_palettes)
        chosen_palette = getattr(px.colors.qualitative, chosen_palette_name)
        print(f"Randomly selected qualitative palette: {chosen_palette_name}")

    # Ensure we get exactly n_colors, cycling if n_colors > len(palette)
    # or truncating if n_colors < len(palette)
    colors = [chosen_palette[i % len(chosen_palette)] for i in range(n_colors)]
    return colors


In [10]:
dataset = load_dataset("billingsmoore/text-clustering-example-data")
data = dataset['train'].to_pandas()
print(data.topic.unique())

['cars and trucks' 'cats and dogs' 'condos and houses'
 'ducks and songbirds' 'apples and oranges' 'religion and spirituality'
 'yoga and meditation' 'football and basketball' 'plants and gardening'
 'cooking and cuisine']


In [11]:

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(data['text'])
#print(embeddings.shape)

tsne = TSNE(n_components=3,random_state=1021)
tsne_compoents = tsne.fit_transform(embeddings)

kmeans = KMeans(n_clusters=10,init='k-means++',random_state=1021)
cluster = kmeans.fit_predict(tsne_compoents)

data[['tsne_c1','tsne_c2','tsne_c3']] = tsne_compoents
data['cluster'] = cluster

In [23]:
categories

array(['cars and trucks', 'cats and dogs', 'condos and houses',
       'ducks and songbirds', 'apples and oranges',
       'religion and spirituality', 'yoga and meditation',
       'football and basketball', 'plants and gardening',
       'cooking and cuisine'], dtype=object)

In [22]:
categories = data['topic'].unique()
random_colors = get_random_discrete_colors(10)
fig = go.Figure()
for i,(category,color) in enumerate(zip(categories,random_colors)):
    df_filtered = data[data['topic'] == category]
    fig.add_trace(go.Scatter3d(
        x=df_filtered['tsne_c1'],
        y=df_filtered['tsne_c2'],
        z=df_filtered['tsne_c3'],
        mode='markers',
        text = df_filtered['text'],
        marker=dict(
            color=color, # Assign color based on category
            opacity=0.7
        ),
        name=category, # This name appears in the legend
        showlegend=True # Ensure legend entry is shown (default is True if name is provided)
    ))

    fig.add_trace(go.Scatter3d(
        x=[kmeans.cluster_centers_[i,0]],
        y=[kmeans.cluster_centers_[i,1]],
        z=[kmeans.cluster_centers_[i,2]],
        mode='markers+text',
        
        text = f"{category} center",
        marker=dict(
            color=color, # Assign color based on category
            opacity=0.7,
            symbol='cross',
        ),
        name=f"{category} center",
        showlegend=False

    ))

fig.update_layout(
    title='Scatter Plot with Legend by Color (go)',
    legend_title='Data Categories' # Customize legend title
)


fig.write_html("./output/cluster.html")

Randomly selected qualitative palette: Light24


In [57]:
category

'apples and oranges'